# VisClick — Colab setup

Run cells **top to bottom**. Authorise Google Drive when prompted.

- **GitHub** → code at `/content/visclick`
- **Drive** `MyDrive/visclick` → datasets + weights (persists after disconnect)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
if not os.path.isdir("/content/visclick"):
    subprocess.run(["git", "clone", REPO, "/content/visclick"], check=True)
else:
    subprocess.run("cd /content/visclick && git pull --rebase", shell=True, check=False)
%cd /content/visclick


In [ ]:
import subprocess
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("GPU check:", e)


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.makedirs(DRIVE, exist_ok=True)
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


In [ ]:
!pip install -q ultralytics==8.* opencv-python "huggingface_hub[cli]>=0.20" tqdm matplotlib pandas


## 05 — Eval, val metrics, export ONNX

`model.val()` on desktop + optional UI-Vision; `export` for Windows bot.


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.environ["VISCLICK_DRIVE"] = DRIVE
!python scripts/patch_colab_configs.py


In [ ]:
from ultralytics import YOLO
import os
DRIVE = "/content/drive/MyDrive/visclick"
# Point to your best run
pt = f"{DRIVE}/weights/experiments/M3_desktop/weights/best.pt"  # adjust name
if not os.path.isfile(pt):
    pt = f"{DRIVE}/weights/baseline_source/best_source_v8s.pt"
m = YOLO(pt)
# Desktop set
d = m.val(data="configs/yolo_desktop_finetune_colab.yaml", split="test")
print(d)


In [ ]:
# Optional: UI-Vision (if yolo config prepared for that data — otherwise skip)
# m.val(data="configs/uivision_data.yaml", ...)
print("Add UI-Vision data.yaml when ready, or use predict() on sample images from ui_vision folder.")


In [ ]:
# Export ONNX (copy to Windows for bot)
from ultralytics import YOLO
import os
DRIVE = "/content/drive/MyDrive/visclick"
pt = f"{DRIVE}/weights/experiments/M3_desktop/weights/best.pt"
m = YOLO(pt) if os.path.isfile(pt) else YOLO(f"{DRIVE}/weights/baseline_source/best_source_v8s.pt")
m.export(format="onnx", opset=12, simplify=True)
print("best.onnx next to best.pt; download to PC into local weights/visclick.onnx")
